# OAI Subject Summary
This notebook illustratse how to use the eluxemourgensia digital collectio and the OAI API ([Open Archives Initiative Protocol for Metadata Harvesting](https://www.openarchives.org/pmh/)) to display the subjects covered by the newspapers in the eLuxemburgensi collectionn
 First, w retrieve the newspaper collection from eluxemburgensia.
For each newspaper, we make a call to the OAI API and retrieve the subject categories. 
These are store and counted in order to display how many newspapers cover the given subject.
The user can then choose one of the subjects and the list of newspapers covering that subject are displayed with a link to the eluxemburgensia collection..

## Requirements
* Python 3.12
* [requests](https://pypi.org/project/requests/): HTTP library to run HTTP requests
* [pandas](https://pandas.pydata.org/): format the output into tabular layout
* [xml minidom](https://docs.python.org/3/library/xml.dom.minidom.html): converting XML received from the OAI call into a Document Object Model interface

In [ ]:
%pip install requests
%pip install pandas

In [ ]:
import requests
from xml.dom.minidom import parse
import xml.dom.minidom as minidom
import pandas as pd

In [ ]:
# get the BnL eluxembourgensia collection
elux_collection = requests.get("https://viewer.eluxemburgensia.lu/api/viewer2/cms/v2/digitalcollections")
elux_collection = elux_collection.json()
subject_list = {}
detailed_list = {}
detailed_list_key = 1

In [ ]:
# A function to get the data from a subfield
def get_subfield_data(subfields):
    for subfield in subfields:
        for childNode in subfield.childNodes:
            return childNode.data

In [ ]:
# A function to add 1 to the count of an entry in a dict
# if no entry, add it to the dict with a count of 1
def add_subject(data):
    if (data in subject_list):
        num = subject_list[data]
        num += 1
        subject_list.update({data : num})
    else:
        subject_list[data]=1

In [ ]:
# subject data is found in 6 different attributes
subject_tag_values = ('650', '651', '652', '653', '654', '655') 
for newspaper in elux_collection["data"]:
    # retrieve the information required about the newspaper
    newspaper_paperid = newspaper["paperid"]
    newspaper_linkaz = newspaper["az_url"]
    newspaper_title = newspaper["title"]
    newspaper_link = "https://persist.lu/" + newspaper["ark"]
    startdate = newspaper["startdate"]
    try:
        enddate = newspaper["enddate"]
    except:
        enddate = ""

    # Find the positions of the start and end of the identifier required for the OAI call. 
    start = "docid=alma"
    start_index = newspaper_linkaz.find(start)
    end_index = newspaper_linkaz.find("&")

    # if there is an identifier, then make the OAI call
    if start_index != -1 and end_index != -1:
        
        # Create the OAI ID for the newspaper
        # Extract the text between “start” and “end” using slice operator
        newspaper_oai_id = "oai:alma.352LUX_BIBNET_NETWORK:" + newspaper_linkaz[start_index + len(start):end_index].strip()
        newspaper_oai_url = "https://oai.bibnet.lu/view/oai/352LUX_BIBNET_NETWORK/request?verb=GetRecord&metadataPrefix=marc21&identifier=" + newspaper_oai_id
        newspaper_oai_data = requests.get(newspaper_oai_url)

        # Open XML document using minidom parser
        DOMTree = minidom.parseString(newspaper_oai_data.text)
        oai_pmh = DOMTree.documentElement
        datafields = oai_pmh.getElementsByTagName("datafield")
        # loop through the data fields, looking for the tags for the subject values
        for datafield in datafields:
            if datafield.hasAttribute('tag'):
                if datafield.getAttribute('tag') in subject_tag_values:
                    # we have a subject attribute.
                    # store the subject data and a count of the number of newspapers
                    subfields = datafield.getElementsByTagName("subfield")  
                    subject = get_subfield_data(subfields)
                    add_subject(subject)
                    # store the detailed information for use later
                    detailed_list[detailed_list_key] = {'Title' : newspaper_title, 'Subject' : subject, 'Start Date' : startdate, 'End Date' : enddate, 'Link' : newspaper_link}
                    detailed_list_key += 1

In [ ]:
print("\nSubjects covered by newspapers in the eLuxemburgensia collection and the number of newspapers for the subject:\n")

# display all the rows in the table - otherwise, some rows are hidden
pd.set_option('display.max_rows', None)

# sort the list by the number of newspapers, descending
subject_list_sorted = {k: v for k, v in sorted(subject_list.items(), key=lambda item: item[1], reverse=True)}

# create a data frame for display purposes
df = pd.DataFrame(subject_list_sorted.items(), columns=["Subject", "Count"])
dfStyler = df.style.set_properties(subset=["Subject"],**{'text-align': 'left'})
dfStyler.set_table_styles([dict(selector='th', props=[('text-align', 'left')])])

In [ ]:
# Request an entry number from the user
chosen_entry = ''
while (chosen_entry=='' or chosen_entry.isdigit() == False):
    chosen_entry = input("Enter the number of the subject for which you wish more information: ")
    if '' == chosen_entry or chosen_entry.isdigit() == False :
        print('Please enter a number.')

In [ ]:
# get the row of the chosen entry
specific_row = df.iloc[int(chosen_entry)]

In [ ]:
# Find all keys with the given value
# Empty list to store matching keys
chosen_items = []

# Loop through dictionary items and find all entries where the subject of the newspaper is the one chosen by the user
for key, value in detailed_list.items():
    if value['Subject'] == specific_row['Subject']:
        chosen_item = {'Title' : value['Title'], 'Subject' : value['Subject'], 'Start Date' : value['Start Date'], 'End Date' : value['End Date'], 'Link' : value['Link']}
        chosen_items.append(chosen_item)

In [ ]:
print("\nNewspapers covering the subject '" + specific_row['Subject'] + "'\n")

# function to make the URL link clickable
def make_clickable(val):
    return f'<a target="_blank" href="{val}">{val}</a>'

# create a data frame with the list of chosen items
chosen_df = pd.DataFrame(chosen_items, columns=["Title", "Subject", "Start Date", "End Date", "Link"])
dfStyler = chosen_df.style.set_properties(**{'text-align': 'left'})
dfStyler.set_table_styles([dict(selector='th', props=[('text-align', 'left')])])
dfStyler.format({'Link': make_clickable})